# Analyse Descriptive des Logs Sécurité

Ce notebook présente une analyse descriptive des logs de sécurité brutes. 
Les étapes suivantes sont réalisées :
1. Chargement des données
2. Classement des règles
3. Histogramme des protocoles
4. Top règles par protocole (UDP/TCP)
5. Analyse approfondie TCP (Règles vs Ports vs Actions)
6. Graphiques de sécurité complémentaires

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de l'affichage
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Chargement des données

In [ ]:
# Chemin vers le fichier parquet
file_path = '../data/brute/logs/logs_export.parquet'

# Chargement
df = pd.read_parquet(file_path)

# Aperçu des données
print(df.info())
df.head()

## 2. Classement des règles les plus utilisées

In [ ]:
# Identification de la colonne règle (adapter si nécessaire, ex: 'policyid', 'rule_name', 'rule_id')
# Hypothèse : colonne 'policyid' ou 'rule_id'. On vérifiera avec df.columns
# Pour l'instant, je vais chercher une colonne probable

possible_rule_cols = ['policyid', 'rule_id', 'rule_name', 'rule']
rule_col = next((col for col in possible_rule_cols if col in df.columns), None)

if rule_col:
    top_rules = df[rule_col].value_counts().head(20)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x=top_rules.index.astype(str), y=top_rules.values, palette='viridis')
    plt.title('Top 20 des règles les plus utilisées')
    plt.xlabel('ID de la règle')
    plt.ylabel('Nombre d\'occurrences')
    plt.xticks(rotation=45)
    plt.show()
    
    print("Classement des règles :")
    print(top_rules)
else:
    print("Colonne de règle non trouvée. Colonnes disponibles :", df.columns)

## 3. Histogramme des protocoles (Déduction TCP/UDP)

In [ ]:
# Fonction pour déduire le protocole si non explicite
def deduce_protocol(row):
    # Si une colonne protocole existe déjà et est explicite
    if 'proto' in row and pd.notna(row['proto']): 
        p = str(row['proto']).lower()
        if 'tcp' in p: return 'TCP'
        if 'udp' in p: return 'UDP'
        if 'icmp' in p: return 'ICMP'
    
    # Déduction basée sur le port (service) si nécessaire
    # Colonne probable : 'dst_port', 'dest_port', 'service'
    port = None
    if 'dst_port' in row: port = row['dst_port']
    elif 'dest_port' in row: port = row['dest_port']
    
    # Logique simplifiée (TCP par défaut pour HTTP/HTTPS/SSH etc, UDP pour DNS/NTP etc)
    # Ceci est une approximation si le protocole n'est pas clair
    # Ports TCP communs : 80, 443, 22, 21, 23, 25, 110, 143, 3389
    # Ports UDP communs : 53, 67, 68, 123, 161, 500, 4500
    
    try:
        port = int(port)
        if port in [53, 67, 68, 123, 161, 500, 4500]: return 'UDP'
        # Par défaut, on peut considérer TCP pour la plupart des services web/admin
        # Ou retourner 'Unknown' si incertain
        return 'TCP' 
    except:
        return 'Autre/Inconnu'

# Application de la déduction
# Note: Si une colonne 'proto' existe avec des numéros (6=TCP, 17=UDP), il faut adapter.
# Je vais vérifier si 'proto' existe.
if 'proto' in df.columns:
    # Si ce sont des nombres (IANA protocol numbers)
    # 1=ICMP, 6=TCP, 17=UDP
    df['protocol_deduced'] = df['proto'].map({6: 'TCP', 17: 'UDP', 1: 'ICMP'}).fillna('Autre')
    # Si ce sont des chaines, on nettoie
    if df['protocol_deduced'].iloc[0] == 'Autre' and isinstance(df['proto'].iloc[0], str):
         df['protocol_deduced'] = df.apply(deduce_protocol, axis=1)
else:
    df['protocol_deduced'] = df.apply(deduce_protocol, axis=1)

# Histogramme
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='protocol_deduced', order=df['protocol_deduced'].value_counts().index, palette='Set2')
plt.title('Répartition des Protocoles (Déduits)')
plt.ylabel('Nombre de logs')
plt.xlabel('Protocole')
plt.show()

## 4. Top 10 des règles UDP et Top 5 des règles TCP

In [ ]:
if rule_col:
    # Top 10 UDP
    top_udp = df[df['protocol_deduced'] == 'UDP'][rule_col].value_counts().head(10)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_udp.index.astype(str), y=top_udp.values, palette='Blues_r')
    plt.title('Top 10 règles - UDP')
    plt.show()
    print("Top 10 UDP:\n", top_udp)
    
    # Top 5 TCP
    top_tcp = df[df['protocol_deduced'] == 'TCP'][rule_col].value_counts().head(5)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_tcp.index.astype(str), y=top_tcp.values, palette='Reds_r')
    plt.title('Top 5 règles - TCP')
    plt.show()
    print("Top 5 TCP:\n", top_tcp)

## 5. Rapprochement Règles vs Ports Destination vs Action (TCP uniquement)

In [ ]:
# Colonnes probables pour port destination et action
dst_port_col = next((col for col in ['dst_port', 'dest_port', 'service', 'port'] if col in df.columns), None)
action_col = next((col for col in ['action', 'status'] if col in df.columns), None)

if rule_col and dst_port_col and action_col:
    # Filtrer sur TCP
    df_tcp = df[df['protocol_deduced'] == 'TCP'].copy()
    
    # On regroupe par Règle, Port, et Action
    analysis_tcp = df_tcp.groupby([rule_col, dst_port_col, action_col]).size().reset_index(name='count')
    
    # On trie par nombre d'occurrences pour voir les plus fréquents
    analysis_tcp = analysis_tcp.sort_values(by='count', ascending=False).head(20)
    
    print("Top 20 Associations : Règle - Port - Action (TCP)")
    print(analysis_tcp)
    
    # Visualisation (Bubble plot ou Heatmap simplifié)
    plt.figure(figsize=(14, 8))
    # Création d'une colonne combinée pour l'axe X
    analysis_tcp['rule_port'] = analysis_tcp[rule_col].astype(str) + " : " + analysis_tcp[dst_port_col].astype(str)
    
    sns.scatterplot(data=analysis_tcp, x='rule_port', y='count', hue=action_col, size='count', sizes=(100, 1000))
    plt.xticks(rotation=90)
    plt.title("Analyse TCP : Règles et Ports par Action")
    plt.tight_layout()
    plt.show()

## 6. Graphiques de sécurité complémentaires

In [ ]:
# 1. Distribution des Actions (Accept/Deny/Drop)
if action_col:
    plt.figure(figsize=(8, 8))
    df[action_col].value_counts().plot.pie(autopct='%1.1f%%', startangle=90, cmap='Pastel1')
    plt.title('Distribution des Actions de Sécurité')
    plt.ylabel('')
    plt.show()

# 2. Top IPs Sources (si colonne src_ip existe)
src_ip_col = next((col for col in ['src_ip', 'source_ip', 'srcip'] if col in df.columns), None)
if src_ip_col:
    top_src = df[src_ip_col].value_counts().head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_src.values, y=top_src.index, palette='magma')
    plt.title('Top 10 Adresses IP Sources')
    plt.xlabel('Nombre de requêtes')
    plt.show()